In [ ]:
import pandas as pd
import os
#React Agent
from typing import Annotated, Sequence, TypedDict, Optional, Dict, Any
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage # The foundational class for all message
from langchain_core.messages import ToolMessage # Passes data back to LLM after it calls
from langchain_core.messages import SystemMessage # Message for providing instructions to
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langchain_tavily import TavilySearch
import base64
load_dotenv()

class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

graph_data = StateGraph(State)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
notebook_env: Dict[str, Any] = {}
@tool
def file_reader(csv_df):
    """This is to read the panda file/dataframe."""
    try:
        name, ext = os.path.splitext(csv_df)
        print(ext)  # Output: .gz
        if ext == ".csv":
            panda_df = pd.read_csv(csv_df)
        elif ext == ".json":
            panda_df = pd.read_json(csv_df)
        elif ext == ".xlsx":
            panda_df = pd.read_excel(csv_df)
        elif ext == ".parquet":
            panda_df = pd.read_parquet(csv_df)
        elif ext == ".sql":
            panda_df = pd.read_sql(csv_df)
        elif ext == ".xml":
            panda_df = pd.read_xml(csv_df)
        else:
            return "File Type not supported, ask to use different file format (csv, json, xlsx, parquet, sql, xml). Please don't continue further"
        print(panda_df.head(5))
        no_nulls = f"This is the amount of nulls: {panda_df.isnull().sum()}"
        df_row, df_col = panda_df.shape
        row_col_info = f"This is the amount of rows: {df_row} and this is the amount of cols: {df_col}"
        df_preview = panda_df.head(df_row)
        preview_info = f"These are the first 5 rows of the dataframe {df_preview.to_string()}"
        df_types = panda_df.dtypes
        types_info = f"These are the column infos (incl. dtypes): {df_types.to_string()}"
        no_nas = panda_df.isna().sum()
        nas_info = f"This is the amount of NAs: {no_nas}"

        report = row_col_info + no_nulls + preview_info + types_info + nas_info
        #print(report)
        return report
    
    
    except Exception as e:
        return f"Error profiling dataset: {str(e)}"


@tool
def code_executor(python_code):
    """Executes python code (incl. visualizations, etc)"""
    try:
        import matplotlib
        matplotlib.use('Agg')
        exec(python_code, globals(), notebook_env)
        if os.path.exists('data_analyzer.png'):
            return "Code executed successfully. A visualization was saved to 'data_analyzer.png.'."
        return "Code executed successfully. Output variables or data aggregations updated."
    
    except Exception as e:
        return f"Error executing code: {str(e)}"
    

tools = [file_reader, code_executor]


from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
system_prompt = (
    "You are an expert Data Analyst AI Agent operating inside a Jupyter Notebook environment.\n\n"
    "Your Workflow:\n"
    "1. Start by using 'metadata_profiler' on the user's file path to inspect the data.\n"
    "2. Formulate a plan to address the user's query.\n"
    "3. Use 'execute_data_analysis_code' to write pandas code, filter, aggregate, or build charts.\n"
    "4. Crucial: If you are making a graph, always save it using `plt.savefig('data_analyzer.png')`.\n"
    "5. If your code throws a syntax error or a KeyError, analyze the error output, fix your mistake, and try again.\n"
    "6. Finish by explaining your mathematical or analytical conclusions to the user."
)

# Compile the ReAct agent with notebook memory
memory = InMemorySaver()
analyst_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
    checkpointer=memory
)
# Create a dummy dataset inside the notebook directory
df_dummy = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'],
    'Sales': [12000, 15000, 14000, 19000, 22000, 26000],
    'Expenses': [9000, 9500, 10000, 11000, 10500, 12000]
})
df_dummy.to_csv('company_performance.csv', index=False)

# Config containing a unique thread id for the session memory
config = {"configurable": {"thread_id": "notebook_session_1"}}

import numpy as np
# Creating a 12-month messy dataset
data = {
    'Date': pd.date_range(start='2025-01-01', periods=12, freq='ME').strftime('%Y-%m-%d'),
    'Category': ['Electronics', 'Clothing', 'Home', 'Electronics', 'Clothing', 'Home', 
                 'Electronics', 'Clothing', 'Home', 'Electronics', 'Clothing', 'Home'],
    'Units_Sold': [150, 320, np.nan, 180, 410, 290, 210, np.nan, 310, 250, 490, 350], # Contains NaNs
    'Revenue': [15000, 6400, 8500, 18000, 8200, 5800, 21000, 9100, 6200, 25000, 9800, 7100],
    'Region': ['North', 'East', 'West', 'South', 'North', 'East', 'West', 'South', 'North', 'East', 'West', 'South']
}

df_test = pd.DataFrame(data)
df_test.to_csv('global_retail_sales.csv', index=False)
print("Test dataset 'global_retail_sales.csv' created successfully!")
# Invoke the agent

import numpy as np
import pandas as pd

# Base dictionary matching our messy, 12-month format
messy_data = {
    'Date': pd.date_range(start='2025-01-01', periods=12, freq='ME').strftime('%Y-%m-%d'),
    'Category': ['Electronics', 'Clothing', 'Home', 'Electronics', 'Clothing', 'Home', 
                 'Electronics', 'Clothing', 'Home', 'Electronics', 'Clothing', 'Home'],
    'Units_Sold': [150, 320, np.nan, 180, 410, 290, 210, np.nan, 310, 250, 490, 350],
    'Revenue': [15000, 6400, 8500, 18000, 8200, 5800, 21000, 9100, 6200, 25000, 9800, 7100],
    'Region': ['North', 'East', 'West', 'South', 'North', 'East', 'West', 'South', 'North', 'East', 'West', 'South']
}
df_base = pd.DataFrame(messy_data)

# 1. Export to JSON (orient='records' formats it like a clean array of objects)
df_base.to_json('global_sales_json.json', orient='records', indent=4)
print("Created: 'global_sales_json.json'")

# 2. Export to Feather (Requires 'pyarrow' or 'fastparquet' installed)
# Note: Feather files don't support period datatypes natively, so strings or datetimes are best.
df_base.to_feather('global_feather_sales.feather')
print("Created: 'global_feather_sales.feather'")
while True:
    user_query = input("Enter query  (type exit, to exit): ")
    if user_query.lower() == "exit":
        break
    inputs = {"messages": [("user", user_query)]}

    for chunk in analyst_agent.stream(inputs, config, stream_mode="values"):
        # Print agent thoughts and tool interactions as they stream in
        last_message = chunk["messages"][-1]
        last_message.pretty_print()
        #